In [1]:
import pandas as pd
import os

In [46]:
df_medium_valid = pd.read_csv("boxoban-astar-solutions/medium_valid.csv")
filtered = df_medium_valid[
    df_medium_valid["Steps"] != "INCORRECT_SOLUTION_FOUND"
].copy()
filtered = filtered[filtered["Actions"] != "SEARCH_STATE_FAILED"].copy()


len(filtered)


49848

In [47]:
def load_level_by_id(path: str, level_id: str) -> str:
    """
    path: path to the .txt level file
    level_id: stringified triple, e.g. "000", "014"

    Returns:
        Level grid as a single string (with newlines)
    """
    target = int(level_id)  # "000" -> 0
    current_id = None
    collecting = False
    level_lines = []

    with open(path, "r") as f:
        for line in f:
            line = line.rstrip("\n")

            # Level header
            if line.startswith(";"):
                # Stop if we were collecting and hit next level
                if collecting:
                    break

                # Parse level number
                try:
                    current_id = int(line[1:].strip())
                except ValueError:
                    current_id = None

                collecting = (current_id == target)
                continue

            # Collect level lines
            if collecting:
                level_lines.append(line)

    if not level_lines:
        raise ValueError(f"Level {level_id} not found in file")

    return "\n".join(level_lines)

level = load_level_by_id("boxoban-levels/medium/valid/000.txt", "000")
print(level)


##########
#       ##
# $    ###
#. ## . ##
# ####  ##
#   #   ##
#$$ #. ###
# # # $@##
# .    ###
##########



In [48]:
from typing import Set, Tuple

def parse_sokoban_level(level_str: str):
    """
    Parses a 10x10 Sokoban level string.

    Returns:
        walls  : Set[(r, c)]
        boxes  : Set[(r, c)]
        goals  : Set[(r, c)]
        player : (r, c)
    """
    walls: Set[Tuple[int, int]] = set()
    boxes: Set[Tuple[int, int]] = set()
    goals: Set[Tuple[int, int]] = set()
    player = None

    rows = level_str.split("\n")

    for r, row in enumerate(rows):
        for c, ch in enumerate(row):
            if ch == "#":
                walls.add((r, c))

            elif ch == "$":
                boxes.add((r, c))

            elif ch == ".":
                goals.add((r, c))

            elif ch == "@":
                player = (r, c)

            elif ch == "*":          # box on goal
                boxes.add((r, c))
                goals.add((r, c))

            elif ch == "+":          # player on goal
                player = (r, c)
                goals.add((r, c))

    if player is None:
        raise ValueError("No player found in level")

    return walls, boxes, goals, player

In [49]:
walls, boxes, goals, player = parse_sokoban_level(level)
action_map = [(-1,0),(0,1),(1,0),(0,-1)]

actions = filtered.head(1)["Actions"].item()

for action in actions:
    if tuple(sorted(boxes)) == tuple(sorted(goals)):
        print("solved")
        break
    dy, dx = action_map[int(action)]
    new_pos_player = (player[0]+dy, player[1]+dx)
    if new_pos_player in walls:
        continue
    if new_pos_player in boxes:
        new_pos_box = (new_pos_player[0]+dy, new_pos_player[1]+dx)
        if new_pos_box in boxes:
            continue
        boxes.remove(new_pos_player)
        boxes.add(new_pos_box)
    player = new_pos_player

if tuple(sorted(boxes)) == tuple(sorted(goals)):
    print("solved")

solved


In [56]:
my_str = str(filtered.head(1)["File"].item())
while len(my_str) < 3:
    my_str = "0"+my_str
print(my_str)

000
